In [3]:
import os
import re
import nibabel as nib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import random
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from scipy.stats import norm
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import OneCycleLR
from util import seed_everything

# Config
MNI_DIR = './data/IXI_extracted/'
CSV_PATH = './data/IXI_extracted/IXI.csv'
VOL_SIZE = 96
N_COMP = 128
FAST_EPOCHS = 20
FAST_LR = 1e-3
BATCH_SIZE = 8
device = torch.device("cuda:5" if torch.cuda.is_available() else "cpu")

seed_everything(0)

# Models
class SpectralViT(nn.Module):
    def __init__(self, n_inputs, n_heads=2, embed_dim=16, n_layers=4):
        super().__init__()
        ranks = torch.arange(1, n_inputs + 1, dtype=torch.float32)
        self.rank_weights = nn.Parameter(1.0 / ranks)
        self.input_proj = nn.Linear(1, embed_dim)
        self.pos_embed = nn.Parameter(torch.randn(n_inputs, 1, embed_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=n_heads, dim_feedforward=embed_dim * 2, dropout=0.1)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.mlp_head = nn.Sequential(nn.LayerNorm(embed_dim), nn.Linear(embed_dim, 1))

    def forward(self, x):
        x = x * self.rank_weights
        x = self.input_proj(x.unsqueeze(-1)).transpose(0, 1)
        x = x + self.pos_embed
        x = self.transformer(x)
        x = x.mean(dim=0)
        return self.mlp_head(x).squeeze(-1)

class SpatialViT(nn.Module):
    def __init__(self, vol_size=96, patch_size=12, embed_dim=128, n_heads=4, n_layers=2):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches = (vol_size // patch_size) ** 3
        self.proj = nn.Linear(patch_size ** 3, embed_dim)
        self.pos_embed = nn.Parameter(torch.randn(self.n_patches + 1, 1, embed_dim) * 0.02)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=n_heads, dim_feedforward=embed_dim*2, dropout=0.2)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.mlp_head = nn.Sequential(nn.LayerNorm(embed_dim), nn.Linear(embed_dim, 1))

    def forward(self, x):
        b = x.shape[0]
        p = self.patch_size
        x = x.unfold(2, p, p).unfold(3, p, p).unfold(4, p, p).contiguous().view(b, self.n_patches, -1)
        x = self.proj(x).transpose(0, 1)
        cls_tokens = self.cls_token.expand(1, b, -1)
        x = torch.cat((cls_tokens, x), dim=0)
        x = x + self.pos_embed
        x = x + self.pos_embed
        x = self.transformer(x)
        return self.mlp_head(x[0]).squeeze(-1)

# Data
def load_ixi_data():
    df = pd.read_csv(CSV_PATH)
    id_col = [c for c in df.columns if 'ID' in c.upper()][0]
    sex_col = [c for c in df.columns if 'SEX' in c.upper()][0]
    sex_lookup = dict(zip(df[id_col].astype(int), df[sex_col].map({1: 0, 2: 1})))
    files = sorted([f for f in os.listdir(MNI_DIR) if f.endswith('.nii.gz')])
    vols, labels = [], []
    for f in tqdm(files, desc="Loading Data"):
        match = re.search(r'(\d+)', f)
        if match and int(match.group(1)) in sex_lookup:
            img = nib.load(os.path.join(MNI_DIR, f)).get_fdata()
            c = np.array(img.shape) // 2
            r = VOL_SIZE // 2
            crop = img[c[0]-r:c[0]+r, c[1]-r:c[1]+r, c[2]-r:c[2]+r]
            if crop.shape == (VOL_SIZE, VOL_SIZE, VOL_SIZE):
                crop = (crop - np.mean(crop)) / (np.std(crop) + 1e-8)
                vols.append(crop.astype(np.float32))
                labels.append(sex_lookup[int(match.group(1))])
    return np.array(vols), np.array(labels).astype(np.float32)

X, Y = load_ixi_data()
X_flat = X.reshape(len(X), -1)

# Memory management
PHYSICAL_BATCH_SIZE = 2 
EFFECTIVE_BATCH_SIZE = 8 
ACCUMULATION_STEPS = EFFECTIVE_BATCH_SIZE // PHYSICAL_BATCH_SIZE

# Grid
patch_candidates = [8, 12, 16]
selection_results = {}

for p_size in patch_candidates:
    print(f"Testing Spatial ViT (patch_size={p_size})...")
    kf = KFold(n_splits=5, shuffle=True, random_state=0)
    fold_aucs = []
    
    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
        # Clear CUDA memory before every fold
        torch.cuda.empty_cache()
        
        tr_vol = torch.from_numpy(X[train_idx]).unsqueeze(1).float()
        tr_y   = torch.from_numpy(Y[train_idx]).float()
        ts_vol = torch.from_numpy(X[test_idx]).unsqueeze(1).float().to(device)
        ts_y   = torch.from_numpy(Y[test_idx]).float().to(device)
        
        # Use the small physical batch size
        loader = DataLoader(TensorDataset(tr_vol, tr_y), batch_size=PHYSICAL_BATCH_SIZE, shuffle=True)
        
        model = SpatialViT(vol_size=VOL_SIZE, patch_size=p_size).to(device)
        opt = optim.AdamW(model.parameters(), lr=FAST_LR)
        sched = OneCycleLR(opt, max_lr=FAST_LR, steps_per_epoch=len(loader)//ACCUMULATION_STEPS, epochs=FAST_EPOCHS)
        crit = nn.BCEWithLogitsLoss()
        
        for _ in range(FAST_EPOCHS):
            model.train()
            for i, (b_vol, b_y) in enumerate(loader):
                b_vol, b_y = b_vol.to(device), b_y.to(device)
                
                # Forward pass
                logits = model(b_vol)
                loss = crit(logits, b_y) / ACCUMULATION_STEPS
                loss.backward()
                
                # Only update weights every ACCUMULATION_STEPS
                if (i + 1) % ACCUMULATION_STEPS == 0:
                    opt.step()
                    sched.step()
                    opt.zero_grad()
        
        model.eval()
        with torch.no_grad():
            # Process test set in smaller groups to avoid memory error
            test_probs = []
            for chunk in torch.split(ts_vol, PHYSICAL_BATCH_SIZE):
                test_probs.append(torch.sigmoid(model(chunk)))
            probs = torch.cat(test_probs).cpu().numpy()
            fold_aucs.append(roc_auc_score(ts_y.cpu(), probs))
            
    selection_results[p_size] = np.mean(fold_aucs)

# Print
best_p = max(selection_results, key=selection_results.get)
print(f"Best Patch Size: {best_p}")

Loading Data:   0%|          | 0/581 [00:00<?, ?it/s]

Loading Data: 100%|██████████| 581/581 [01:36<00:00,  6.03it/s]


Testing Spatial ViT (patch_size=8)...
Testing Spatial ViT (patch_size=12)...
Testing Spatial ViT (patch_size=16)...
Best Patch Size: 12


In [4]:
import os
import re
import nibabel as nib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import random
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import OneCycleLR
from util import seed_everything

# Configuration
MNI_DIR = './data/IXI_extracted/'
CSV_PATH = './data/IXI_extracted/IXI.csv'
VOL_SIZE = 96
FAST_EPOCHS = 20
FAST_LR = 1e-3
BATCH_SIZE = len(ts_y)
device = torch.device("cuda:5" if torch.cuda.is_available() else "cpu")

seed_everything(0)

# Models
class SpectralViT(nn.Module):
    def __init__(self, n_inputs, n_heads=2, embed_dim=16, n_layers=4):
        super().__init__()
        ranks = torch.arange(1, n_inputs + 1, dtype=torch.float32)
        self.rank_weights = nn.Parameter(1.0 / ranks)
        self.input_proj = nn.Linear(1, embed_dim)
        self.pos_embed = nn.Parameter(torch.randn(n_inputs, 1, embed_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=n_heads, dim_feedforward=embed_dim * 2, dropout=0.1)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.mlp_head = nn.Sequential(nn.LayerNorm(embed_dim), nn.Linear(embed_dim, 1))

    def forward(self, x):
        # x shape: (batch, n_inputs)
        x = x * self.rank_weights
        x = self.input_proj(x.unsqueeze(-1)).transpose(0, 1) # (seq, batch, embed)
        x = x + self.pos_embed
        x = self.transformer(x)
        x = x.mean(dim=0)
        return self.mlp_head(x).squeeze(-1)

# Data
X_flat = X.reshape(len(X), -1)

# Grid
pca_candidates = [16, 32, 64, 128]
spectral_selection_results = {}

print(f"Starting Selection over PCA components: {pca_candidates}")

for n_comp in pca_candidates:
    print(f"Testing Spectral ViT (n_components={n_comp})...")
    kf = KFold(n_splits=5, shuffle=True, random_state=0)
    fold_aucs = []
    
    for fold, (train_idx, test_idx) in enumerate(kf.split(X_flat)):
        torch.cuda.empty_cache()
        
        # Fit PCA on training data only
        pca_model = PCA(n_components=n_comp, whiten=True).fit(X_flat[train_idx])
        
        # Transform data
        tr_pca = torch.from_numpy(pca_model.transform(X_flat[train_idx])).float()
        tr_y = torch.from_numpy(Y[train_idx]).float()
        ts_pca = torch.from_numpy(pca_model.transform(X_flat[test_idx])).float().to(device)
        ts_y = torch.from_numpy(Y[test_idx]).float().to(device)
        
        loader = DataLoader(TensorDataset(tr_pca, tr_y), batch_size=BATCH_SIZE, shuffle=True)
        
        # Initialize spectral ViT
        model = SpectralViT(n_inputs=n_comp, embed_dim=16).to(device)
        opt = optim.AdamW(model.parameters(), lr=FAST_LR)
        sched = OneCycleLR(opt, max_lr=FAST_LR, steps_per_epoch=len(loader), epochs=FAST_EPOCHS)
        crit = nn.BCEWithLogitsLoss()
        
        # Train
        for _ in range(FAST_EPOCHS):
            model.train()
            for b_pca, b_y in loader:
                b_pca, b_y = b_pca.to(device), b_y.to(device)
                opt.zero_grad()
                loss = crit(model(b_pca), b_y)
                loss.backward()
                opt.step()
                sched.step()
        
        # Evaluate
        model.eval()
        with torch.no_grad():
            probs = torch.sigmoid(model(ts_pca)).cpu().numpy()
            fold_aucs.append(roc_auc_score(ts_y.cpu(), probs))
            
    spectral_selection_results[n_comp] = np.mean(fold_aucs)

# Print
best_n = max(spectral_selection_results, key=spectral_selection_results.get)
print(f"Best PCA Components: {best_n}")

Starting Selection over PCA components: [16, 32, 64, 128]
Testing Spectral ViT (n_components=16)...
Testing Spectral ViT (n_components=32)...
Testing Spectral ViT (n_components=64)...
Testing Spectral ViT (n_components=128)...
Best PCA Components: 128
